fasttext模型对词的内部结构进行了探讨。<br>
如指定单词内子词的长度(如3~6)，每个中心词由其子词级向量之和表示。<br>

字节对编码

In [1]:
import collections 

# 符号词表初始化
# _ 词尾符号, UNK 未知符号
symbols = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n'
            'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '_', '[UNK]']

In [2]:
raw_token_freqs = {'fast_': 4, 'faster_': 3, 'tall_' : 5, 'taller_': 4}
token_freqs = {}
for token, freq in raw_token_freqs.items():
    token_freqs[' '.join(list(token))] = raw_token_freqs[token]
token_freqs

{'f a s t _': 4, 'f a s t e r _': 3, 't a l l _': 5, 't a l l e r _': 4}

In [3]:
def get_max_freq_pair(token_freqs):
    """返回词内出现最频繁的连续符号对"""
    pairs = collections.defaultdict(int)     # 当访问不存在的键时，默认初始化为int(),即0
    for token, freq in token_freqs.items():
        symbols = token.split()              # 列表，每个字符分隔开
        for i in range(len(symbols) - 1):    # 计算连续符号对的出现次数
            # pairs的键是两个连续符号的元组
            pairs[symbols[i], symbols[i+1]] += freq 
    return max(pairs, key=pairs.get)               # 具有最大值的pairs键

In [4]:
def merge_symbols(max_freq_pair, token_freqs, symbols):
    symbols.append(''.join(max_freq_pair))                 # 最频繁的连续符号对，合并成无空格的字符
    new_token_freqs = dict()
    for token, freq in token_freqs.items():
        new_token = token.replace(' '.join(max_freq_pair), # 将之前有空格的，替换成无空格的 
                                  ''.join(max_freq_pair))
        new_token_freqs[new_token] = token_freqs[token]
    return new_token_freqs

In [ ]:
# 合并出现最频繁的连续符号对以生成新符号
num_merges = 10
for i in range(num_merges):
    max_freq_pair = get_max_freq_pair(token_freqs)
    token_freqs = merge_symbols(max_freq_pair, token_freqs, symbols)
    print(f'合并# {i+1}:', max_freq_pair)

合并# 1: ('t', 'a')
合并# 2: ('ta', 'l')
合并# 3: ('tal', 'l')
合并# 4: ('f', 'a')
合并# 5: ('fa', 's')
合并# 6: ('fas', 't')
合并# 7: ('e', 'r')
合并# 8: ('er', '_')
合并# 9: ('tall', '_')
合并# 10: ('fast', '_')


In [6]:
# 多包含了10个从其他符号迭代合并而来的符号
print(symbols)

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'no', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '_', '[UNK]', 'ta', 'tal', 'tall', 'fa', 'fas', 'fast', 'er', 'er_', 'tall_', 'fast_']


In [7]:
# 子词的切分
print(list(token_freqs.keys()))

['fast_', 'fast er_', 'tall_', 'tall er_']


In [ ]:
def segment_BPE(tokens, symbols):
    """使用从一个数据集学习好的子词symbols来切分另一个数据集的词"""
    outputs = []                        # 输出
    for token in tokens:                # 对一个词
        start, end = 0, len(token)
        cur_output = []                 # 当前词的输出
        # 具有符号中可能最长子词的词元段
        while start < len(token) and start < end:
            if token[start: end] in symbols:     # 如果token中出现了symbols中的子词
                cur_output.append(token[start:end])
                start = end                      # 探测后续子词
                end = len(token)
            else:
                end -= 1                        # 从尾部开始缩小搜索范围
        if start < len(token):             # 假如词内有未知字符，end一直缩小直到<=start, 此时while循环结束时，start<len(token)
            cur_output.append('[UNK]')
        outputs.append(' '.join(cur_output))
    return outputs
tokens = ['tallest_', 'fatter_']
print(segment_BPE(tokens, symbols))

['tall e s t _', 'fa t t er_', 'tall e [UNK]']


子词嵌入可以提高罕见词和词典外词的表现质量